In [0]:
%sql
MERGE INTO assaia_catalog.gold.dimClientes AS tgt
USING (
  SELECT
    id                  AS id_cliente,
    nome_completo,
    cpf,
    genero,
    current_timestamp() AS _dt_atualizacao
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
             PARTITION BY id
             ORDER BY _fivetran_synced DESC, _line DESC
           ) AS rn
    FROM assaia_catalog.silver.dClientes
    WHERE id IS NOT NULL
  ) x
  WHERE rn = 1
) src
ON tgt.id_cliente = src.id_cliente
WHEN MATCHED THEN UPDATE SET
  tgt.nome_completo   = src.nome_completo,
  tgt.cpf             = src.cpf,
  tgt.genero          = src.genero,
  tgt._dt_atualizacao = src._dt_atualizacao
WHEN NOT MATCHED THEN INSERT (
  id_cliente, nome_completo, cpf, genero, _dt_atualizacao
) VALUES (
  src.id_cliente, src.nome_completo, src.cpf, src.genero, src._dt_atualizacao
);

In [0]:
%sql
MERGE INTO assaia_catalog.gold.dimProdutos AS tgt
USING (
  SELECT
    id                                  AS id_produto,
    descricao,
    marca,
    categorias,
    sub_categoria,
    CAST(custo AS DECIMAL(18,2))        AS custo,
    CAST(preco_venda AS DECIMAL(18,2))  AS preco_venda,
    CAST(preco_venda - custo AS DECIMAL(18,2)) AS margem_unitaria,
    current_timestamp()                 AS _dt_atualizacao
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
             PARTITION BY id
             ORDER BY _fivetran_synced DESC, _line DESC
           ) AS rn
    FROM assaia_catalog.silver.dProdutos
    WHERE id IS NOT NULL
  ) x
  WHERE rn = 1
) src
ON tgt.id_produto = src.id_produto
WHEN MATCHED THEN UPDATE SET
  tgt.descricao       = src.descricao,
  tgt.marca           = src.marca,
  tgt.categorias      = src.categorias,
  tgt.sub_categoria   = src.sub_categoria,
  tgt.custo           = src.custo,
  tgt.preco_venda     = src.preco_venda,
  tgt.margem_unitaria = src.margem_unitaria,
  tgt._dt_atualizacao = src._dt_atualizacao
WHEN NOT MATCHED THEN INSERT (
  id_produto, descricao, marca, categorias, sub_categoria,
  custo, preco_venda, margem_unitaria, _dt_atualizacao
) VALUES (
  src.id_produto, src.descricao, src.marca, src.categorias, src.sub_categoria,
  src.custo, src.preco_venda, src.margem_unitaria, src._dt_atualizacao
);

In [0]:
%sql
MERGE INTO assaia_catalog.gold.dimGerentes AS tgt
USING (
  SELECT
    id                  AS id_gerente,
    nome_gerente,
    cpf,
    current_timestamp() AS _dt_atualizacao
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
             PARTITION BY id
             ORDER BY _fivetran_synced DESC, _line DESC
           ) AS rn
    FROM assaia_catalog.silver.dGerentes
    WHERE id IS NOT NULL
  ) x
  WHERE rn = 1
) src
ON tgt.id_gerente = src.id_gerente
WHEN MATCHED THEN UPDATE SET
  tgt.nome_gerente    = src.nome_gerente,
  tgt.cpf             = src.cpf,
  tgt._dt_atualizacao = src._dt_atualizacao
WHEN NOT MATCHED THEN INSERT (
  id_gerente, nome_gerente, cpf, _dt_atualizacao
) VALUES (
  src.id_gerente, src.nome_gerente, src.cpf, src._dt_atualizacao
);

In [0]:
%sql
MERGE INTO assaia_catalog.gold.dimLojas AS tgt
USING (
  SELECT
    id                  AS id_loja,
    nome_unidade,
    endereco,
    id_gerente_geral,
    current_timestamp() AS _dt_atualizacao
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
             PARTITION BY id
             ORDER BY _fivetran_synced DESC, _line DESC
           ) AS rn
    FROM assaia_catalog.silver.dLojas
    WHERE id IS NOT NULL
      AND id_gerente_geral IS NOT NULL
  ) x
  WHERE rn = 1
) src
ON tgt.id_loja = src.id_loja
WHEN MATCHED THEN UPDATE SET
  tgt.nome_unidade     = src.nome_unidade,
  tgt.endereco         = src.endereco,
  tgt.id_gerente_geral = src.id_gerente_geral,
  tgt._dt_atualizacao  = src._dt_atualizacao
WHEN NOT MATCHED THEN INSERT (
  id_loja, nome_unidade, endereco, id_gerente_geral, _dt_atualizacao
) VALUES (
  src.id_loja, src.nome_unidade, src.endereco, src.id_gerente_geral, src._dt_atualizacao
);

In [0]:
%sql
INSERT OVERWRITE assaia_catalog.gold.dimTempo
WITH limites AS (
  SELECT
    MIN(to_date(data_venda)) AS dt_min,
    MAX(to_date(data_venda)) AS dt_max
  FROM assaia_catalog.silver.fVendas
  WHERE data_venda IS NOT NULL
),
datas AS (
  SELECT explode(sequence(dt_min, dt_max, interval 1 day)) AS data_calendario
  FROM limites
)
SELECT
  CAST(date_format(data_calendario, 'yyyyMMdd') AS INT)                     AS id_tempo,
  data_calendario                                                           AS data_calendario,
  year(data_calendario)                                                     AS ano,
  month(data_calendario)                                                    AS mes,
  CASE 
    WHEN date_format(data_calendario, 'MMMM')  = 'January' THEN 'Janeiro'
    WHEN date_format(data_calendario, 'MMMM')  = 'February' THEN 'Fevereiro'
    WHEN date_format(data_calendario, 'MMMM')  = 'March' THEN 'Março'
    WHEN date_format(data_calendario, 'MMMM')  = 'April' THEN 'Abril'
    WHEN date_format(data_calendario, 'MMMM')  = 'May' THEN 'Maio'
    WHEN date_format(data_calendario, 'MMMM')  = 'June' THEN 'Junho'
    WHEN date_format(data_calendario, 'MMMM')  = 'July' THEN 'Julho'
    WHEN date_format(data_calendario, 'MMMM')  = 'August' THEN 'Agosto'
    WHEN date_format(data_calendario, 'MMMM')  = 'September' THEN 'Setembro'
    WHEN date_format(data_calendario, 'MMMM')  = 'October' THEN 'Outubro'
    WHEN date_format(data_calendario, 'MMMM')  = 'November' THEN 'Novembro'
    WHEN date_format(data_calendario, 'MMMM')  = 'December' THEN 'Dezembro'
    ELSE ""
  END                                                                       AS nome_mes,
  quarter(data_calendario)                                                  AS trimestre,
  weekofyear(data_calendario)                                               AS semana_ano,
  day(data_calendario)                                                      AS dia_mes,
  dayofweek(data_calendario)                                                AS dia_semana,
CASE 
  WHEN date_format(data_calendario, 'EEEE') = "Monday"    THEN "Segunda-Feira"
  WHEN date_format(data_calendario, 'EEEE') = "Tuesday"   THEN "Terça-Feira"
  WHEN date_format(data_calendario, 'EEEE') = "Wednesday" THEN "Quarta-Feira"
  WHEN date_format(data_calendario, 'EEEE') = "Thursday"  THEN "Quinta-Feira"
  WHEN date_format(data_calendario, 'EEEE') = "Friday"    THEN "Sexta-Feira"
  WHEN date_format(data_calendario, 'EEEE') = "Saturday"  THEN "Sábado"
  WHEN date_format(data_calendario, 'EEEE') = "Sunday"    THEN "Domingo"
  ELSE "" END                                                                AS nome_dia_semana,
  CASE WHEN dayofweek(data_calendario) IN (1, 7) THEN 1 ELSE 0 END   AS fim_de_semana
FROM datas;

In [0]:
%sql
MERGE INTO assaia_catalog.gold.fatoVendas AS tgt
USING (
  SELECT
    v.id_cliente,
    v.id_produto,
    v.id_loja,
    CAST(date_format(to_date(v.data_venda), 'yyyyMMdd') AS INT)          AS id_tempo,
    v.data_venda,
    v.forma_pagamento,
    v.qtd,
    CAST(p.preco_venda AS DECIMAL(18,2))                                 AS valor_unitario,
    CAST(v.qtd * p.preco_venda AS DECIMAL(18,2))                         AS valor_total,
    CAST(v.qtd * p.custo AS DECIMAL(18,2))                               AS custo_total,
    CAST((v.qtd * p.preco_venda) - (v.qtd * p.custo) AS DECIMAL(18,2))   AS margem_total,
    current_timestamp()                                                  AS _dt_atualizacao
  FROM (
    SELECT *
    FROM (
      SELECT *,
             ROW_NUMBER() OVER (
               PARTITION BY id_cliente, id_produto, id_loja, data_venda
               ORDER BY _fivetran_synced DESC, _line DESC
             ) AS rn
      FROM assaia_catalog.silver.fVendas
      WHERE id_cliente IS NOT NULL
        AND id_produto IS NOT NULL
        AND id_loja IS NOT NULL
        AND data_venda IS NOT NULL
        AND qtd > 0
    ) x
    WHERE rn = 1
  ) v
  INNER JOIN assaia_catalog.gold.dimProdutos p
    ON v.id_produto = p.id_produto
  INNER JOIN assaia_catalog.gold.dimClientes c
    ON v.id_cliente = c.id_cliente
  INNER JOIN assaia_catalog.gold.dimLojas l
    ON v.id_loja = l.id_loja
) src
ON  tgt.id_cliente = src.id_cliente
AND tgt.id_produto = src.id_produto
AND tgt.id_loja    = src.id_loja
AND tgt.data_venda = src.data_venda
WHEN MATCHED THEN UPDATE SET
  tgt.id_tempo         = src.id_tempo,
  tgt.forma_pagamento  = src.forma_pagamento,
  tgt.qtd              = src.qtd,
  tgt.valor_unitario   = src.valor_unitario,
  tgt.valor_total      = src.valor_total,
  tgt.custo_total      = src.custo_total,
  tgt.margem_total     = src.margem_total,
  tgt._dt_atualizacao  = src._dt_atualizacao
WHEN NOT MATCHED THEN INSERT (
  id_cliente, id_produto, id_loja, id_tempo, data_venda, forma_pagamento,
  qtd, valor_unitario, valor_total, custo_total, margem_total, _dt_atualizacao
) VALUES (
  src.id_cliente, src.id_produto, src.id_loja, src.id_tempo, src.data_venda, src.forma_pagamento,
  src.qtd, src.valor_unitario, src.valor_total, src.custo_total, src.margem_total, src._dt_atualizacao
);

In [0]:
%sql
MERGE INTO assaia_catalog.gold.fatoVendas AS tgt
USING (
  SELECT
    v.id_cliente,
    v.id_produto,
    v.id_loja,
    CAST(date_format(to_date(v.data_venda), 'yyyyMMdd') AS INT)          AS id_tempo,
    v.data_venda,
    v.forma_pagamento,
    v.qtd,
    CAST(p.preco_venda AS DECIMAL(18,2))                                 AS valor_unitario,
    CAST(v.qtd * p.preco_venda AS DECIMAL(18,2))                         AS valor_total,
    CAST(v.qtd * p.custo AS DECIMAL(18,2))                               AS custo_total,
    CAST((v.qtd * p.preco_venda) - (v.qtd * p.custo) AS DECIMAL(18,2))   AS margem_total,
    current_timestamp()                                                  AS _dt_atualizacao
  FROM (
    SELECT *
    FROM (
      SELECT *,
             ROW_NUMBER() OVER (
               PARTITION BY id_cliente, id_produto, id_loja, data_venda
               ORDER BY _fivetran_synced DESC, _line DESC
             ) AS rn
      FROM assaia_catalog.silver.fVendas
      WHERE id_cliente IS NOT NULL
        AND id_produto IS NOT NULL
        AND id_loja IS NOT NULL
        AND data_venda IS NOT NULL
        AND qtd > 0
    ) x
    WHERE rn = 1
  ) v
  INNER JOIN assaia_catalog.gold.dimProdutos p
    ON v.id_produto = p.id_produto
  INNER JOIN assaia_catalog.gold.dimClientes c
    ON v.id_cliente = c.id_cliente
  INNER JOIN assaia_catalog.gold.dimLojas l
    ON v.id_loja = l.id_loja
) src
ON  tgt.id_cliente = src.id_cliente
AND tgt.id_produto = src.id_produto
AND tgt.id_loja    = src.id_loja
AND tgt.data_venda = src.data_venda
WHEN MATCHED THEN UPDATE SET
  tgt.id_tempo         = src.id_tempo,
  tgt.forma_pagamento  = src.forma_pagamento,
  tgt.qtd              = src.qtd,
  tgt.valor_unitario   = src.valor_unitario,
  tgt.valor_total      = src.valor_total,
  tgt.custo_total      = src.custo_total,
  tgt.margem_total     = src.margem_total,
  tgt._dt_atualizacao  = src._dt_atualizacao
WHEN NOT MATCHED THEN INSERT (
  id_cliente, id_produto, id_loja, id_tempo, data_venda, forma_pagamento,
  qtd, valor_unitario, valor_total, custo_total, margem_total, _dt_atualizacao
) VALUES (
  src.id_cliente, src.id_produto, src.id_loja, src.id_tempo, src.data_venda, src.forma_pagamento,
  src.qtd, src.valor_unitario, src.valor_total, src.custo_total, src.margem_total, src._dt_atualizacao
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS assaia_catalog.gold.martVendasDiariasLoja (
  data_venda             DATE           NOT NULL COMMENT 'Data da venda',
  id_loja                INT            NOT NULL COMMENT 'ID da loja',
  nome_unidade           STRING         NOT NULL COMMENT 'Nome da loja',
  qtd_itens_vendidos     BIGINT         NOT NULL COMMENT 'Quantidade total de itens vendidos',
  faturamento_total      DECIMAL(18,2)  NOT NULL COMMENT 'Faturamento total',
  custo_total            DECIMAL(18,2)  NOT NULL COMMENT 'Custo total',
  margem_total           DECIMAL(18,2)  NOT NULL COMMENT 'Margem total',
  ticket_medio           DECIMAL(18,2)  NOT NULL COMMENT 'Ticket médio por registro analítico',
  _dt_atualizacao        TIMESTAMP      NOT NULL COMMENT 'Data de atualização'
  CONSTRAINT chk_gold_martVendasDiariasLoja_qtd CHECK (qtd_itens_vendidos >= 0)
)
USING DELTA
COMMENT 'Gold: mart analítico de vendas diárias por loja'
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true'
);

In [0]:
%sql
INSERT OVERWRITE assaia_catalog.gold.martVendasDiariasLoja
SELECT
  to_date(data_venda)                                           AS data_venda,
  f.id_loja                                                     AS id_loja,
  l.nome_unidade                                                AS nome_unidade,
  SUM(f.qtd)                                                    AS qtd_itens_vendidos,
  CAST(SUM(f.valor_total) AS DECIMAL(18,2))                     AS faturamento_total,
  CAST(SUM(f.custo_total) AS DECIMAL(18,2))                     AS custo_total,
  CAST(SUM(f.margem_total) AS DECIMAL(18,2))                    AS margem_total,
  CAST(AVG(f.valor_total) AS DECIMAL(18,2))                     AS ticket_medio,
  current_timestamp()                                           AS _dt_atualizacao
FROM assaia_catalog.gold.fatoVendas f
INNER JOIN assaia_catalog.gold.dimLojas l
  ON f.id_loja = l.id_loja
GROUP BY
  to_date(data_venda),
  f.id_loja,
  l.nome_unidade;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS assaia_catalog.gold.martVendasMensaisCategoria (
  ano                   INT            NOT NULL COMMENT 'Ano',
  mes                   INT            NOT NULL COMMENT 'Mês',
  categorias            STRING         NOT NULL COMMENT 'Categoria',
  sub_categoria         STRING                     COMMENT 'Subcategoria',
  qtd_itens_vendidos    BIGINT         NOT NULL COMMENT 'Quantidade total de itens',
  faturamento_total     DECIMAL(18,2)  NOT NULL COMMENT 'Faturamento total',
  custo_total           DECIMAL(18,2)  NOT NULL COMMENT 'Custo total',
  margem_total          DECIMAL(18,2)  NOT NULL COMMENT 'Margem total',
  _dt_atualizacao       TIMESTAMP      NOT NULL COMMENT 'Data de atualização'
)
USING DELTA
COMMENT 'Gold: mart mensal por categoria e subcategoria'
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true'
);

In [0]:
%sql
INSERT OVERWRITE assaia_catalog.gold.martVendasMensaisCategoria
SELECT
  t.ano                                                        AS ano,
  t.mes                                                        AS mes,
  p.categorias                                                 AS categorias,
  p.sub_categoria                                              AS sub_categoria,
  SUM(f.qtd)                                                   AS qtd_itens_vendidos,
  CAST(SUM(f.valor_total) AS DECIMAL(18,2))                    AS faturamento_total,
  CAST(SUM(f.custo_total) AS DECIMAL(18,2))                    AS custo_total,
  CAST(SUM(f.margem_total) AS DECIMAL(18,2))                   AS margem_total,
  current_timestamp()                                          AS _dt_atualizacao
FROM assaia_catalog.gold.fatoVendas f
INNER JOIN assaia_catalog.gold.dimTempo t
  ON f.id_tempo = t.id_tempo
INNER JOIN assaia_catalog.gold.dimProdutos p
  ON f.id_produto = p.id_produto
GROUP BY
  t.ano,
  t.mes,
  p.categorias,
  p.sub_categoria;

In [0]:
%sql
MERGE INTO assaia_catalog.gold.dimEnderecos AS tgt
USING (
  SELECT
    id                  AS id_endereco,
    logradouro,
    bairro,
    cidade,
    estado,
    uf,
    pais,
    current_timestamp() AS _dt_atualizacao
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
             PARTITION BY id
             ORDER BY _fivetran_synced DESC, _line DESC
           ) AS rn
    FROM assaia_catalog.silver.dEnderecos
    WHERE id IS NOT NULL
  ) x
  WHERE rn = 1
) src
ON tgt.id_endereco = src.id_endereco
WHEN MATCHED THEN UPDATE SET
  tgt.logradouro      = src.logradouro,
  tgt.bairro          = src.bairro,
  tgt.cidade          = src.cidade,
  tgt.estado          = src.estado,
  tgt.uf              = src.uf,
  tgt.pais            = src.pais,
  tgt._dt_atualizacao = src._dt_atualizacao
WHEN NOT MATCHED THEN INSERT (
  id_endereco, logradouro, bairro, cidade, estado, uf, pais, _dt_atualizacao
) VALUES (
  src.id_endereco, src.logradouro, src.bairro, src.cidade, src.estado, src.uf, src.pais, src._dt_atualizacao
);